# Part E — Multi-step Tool Calling: Tool Result → Model → Tool Result → Final Answer

**Goal:** Observe a *real* local Qwen3.5-4B model asking for multiple read-only tools, receiving their actual SQLite results, and producing a grounded answer. This section builds directly on Parts A–D, without changing your working backend or reloading the model.

**Prerequisites**
- Part A has created `bookstore.db` and `books_api.py`.
- Part C has created `bookstore_tools.py` (`search_books`, `get_book_rating` only).
- Part D has created `d_tool_protocol.py` and `d_tool_executor.py`, and your single-turn selection test worked.
- Part B's `%run -i qwen_load.py` has already placed `model`, `processor`, and `tokenizer` in this same notebook runtime.

**No installations are required in Part E.** We reuse the existing Qwen model and LangChain tool definitions. Every new Python implementation lives in a file written with `%%writefile`.


## E0 — Predict the tool-call sequence (before running anything)

> **Request:** *Find a sci-fi bestseller under $16 with a rating above 4.5.*

Write your predictions:

1. **What** information can `search_books` return? Does it include review ratings?
2. **Why** might `get_book_rating` be necessary after `search_books`?
3. **How many** model turns are needed if Qwen selects one tool on each turn and then gives a final answer?
4. **Who** selects each tool, and **who** executes it?
5. **What** should happen if the tool returns no matching books?

After executing Part E, compare your prediction to the **actual** trace. The model is *not* guaranteed to follow any predetermined sequence.


## E1 — Implement the transparent agent loop

The loop stores a conversation history with `system`, `user`, `assistant`, and `tool` messages. For each model turn it:

1. Applies the **Qwen3.5 chat template** with the two tool schemas.
2. Runs **real model inference** to obtain a tool request or normal answer.
3. Parses the result with Part D's parser.
4. Invokes **only allowlisted, read-only** Python tools via the Part D executor.
5. Appends structured `assistant.tool_calls` and corresponding `tool` response messages, then asks the model again.
6. Stops when a final answer is produced or a safety/budget limit is reached.

**Important:** A real tool response is placed back in model context; printing the tool result by itself (as in Part D) would **not** create a second model turn. We use the tokenizer's built-in tool-call template rather than hand-assembling XML. [Qwen3.5 chat template](https://huggingface.co/Qwen/Qwen3.5-4B/blob/main/chat_template.jinja).


In [ ]:
%%writefile e_agent_loop.py
"""Part E1 — A transparent, bounded Qwen tool-calling loop.

Use from a Colab notebook; the already-loaded Qwen model is supplied by
`e_run_agent.py`. This module contains NO hard-coded answers/tool decisions.

The model proposes calls; the trusted Python dispatcher executes only the
read-only allowlist established in Parts C/D. This is a teaching loop, not a
production-grade agent orchestration/security framework.
"""
import json
import re
from typing import Callable, Dict, List

from d_tool_protocol import proposed_calls

SYSTEM_PROMPT = (
    "You are a precise bookstore assistant. The bookstore database is the "
    "only source of truth for titles, prices, bestseller status, and ratings. "
    "Available read-only tools: search_books (catalog filters, book IDs, prices, "
    "NO ratings) and get_book_rating (average rating of a book ID). "
    "If the user asks you to find a book satisfying a rating threshold, first "
    "use search_books to identify candidates and then use get_book_rating to "
    "verify ratings before recommending. Never invent ratings, prices, or "
    "book IDs. If no qualifying book is found, say so. "
    "When sufficient verified information is available, stop calling tools and "
    "give a short answer including the book title, price, rating and review count "
    "when relevant. Do not ask to buy or execute purchases: no purchase tool is available."
)


def clean_answer(raw: str) -> str:
    """Remove generation delimiters; don't expose any internal think blocks."""
    text = re.sub(r"<think>.*?</think>", "", raw, flags=re.S)
    if "</think>" in text:  # handles a missing opening tag in a partial response
        text = text.split("</think>", 1)[1]
    for token in ("<|im_end|>", "<|endoftext|>", "<|eot_id|>"):
        text = text.replace(token, "")
    return text.strip()


def initial_history(question: str) -> List[dict]:
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question must contain text")
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question.strip()},
    ]


def add_exchange(history: List[dict], calls: List[dict], results: List[dict]) -> None:
    """Append the Qwen3.5-compatible assistant tool-call and tool-result turns.

    The tokenizer's chat template, not this module, renders them into the
    model-specific <function=...> and <tool_response> wire format.
    """
    if len(calls) != len(results):
        raise ValueError("Every proposed tool call must have a corresponding result")
    history.append({
        "role": "assistant", "content": "",
        "tool_calls": [
            {"type": "function", "function": {
                "name": call["name"], "arguments": call["arguments"]
            }}
            for call in calls
        ],
    })
    for call, result in zip(calls, results):
        history.append({
            "role": "tool", "name": call["name"],
            "content": json.dumps(result, ensure_ascii=False),
        })


def generate_qwen_reply(history: List[dict], model, processor, tokenizer,
                        max_new_tokens: int = 384) -> str:
    """One real local inference; model must already be loaded in the notebook."""
    import torch
    from bookstore_tools import TOOL_SCHEMAS

    inputs = processor.apply_chat_template(
        history,
        tools=TOOL_SCHEMAS,
        enable_thinking=False,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = generated[0][inputs["input_ids"].shape[-1]:]
    return processor.decode(new_tokens, skip_special_tokens=False).strip()


def run_agent(question: str, generate_reply: Callable,
              execute_calls: Callable, *, max_rounds: int = 5,
              max_tool_calls: int = 8, max_new_tokens: int = 384,
              verbose: bool = True) -> Dict:
    """Repeatedly call the model, dispatch its approved tools, and feed results back.

    `generate_reply(history, max_new_tokens)` is an injectable function, so
    we can unit test orchestration WITHOUT claiming a simulated model is Qwen.
    `execute_calls(list_of_calls)` is the Part D safe read-only executor.
    """
    if not 1 <= max_rounds <= 10 or not 1 <= max_tool_calls <= 20:
        raise ValueError("Set 1 <= max_rounds <= 10 and 1 <= max_tool_calls <= 20")
    history = initial_history(question)
    trace = {"question": question, "status": "running", "answer": None,
             "turns": [], "tool_call_count": 0, "messages": history}
    last_signature = None

    if verbose:
        print("USER:", question)
        print("Bounded agent: up to", max_rounds, "model turns;", max_tool_calls, "tool calls.")
    for step in range(1, max_rounds + 1):
        raw = generate_reply(history, max_new_tokens)
        proposed = proposed_calls(raw)
        calls = proposed["calls"]
        turn = {"turn": step, "raw_model_output": raw,
                "selection_status": proposed["status"],
                "requested_tools": calls, "tool_results": []}
        trace["turns"].append(turn)
        if verbose:
            print(f"\n{'='*18} MODEL TURN {step} {'='*18}")
            print("Raw Qwen output:\n", raw)
            print("Parsed requests:", json.dumps(calls, ensure_ascii=False))

        if proposed["status"] == "invalid_tool_call":
            trace["status"] = "invalid_tool_call"
            if verbose:
                print("STOP: malformed tool call; nothing executed on this turn.")
            break
        if not calls:
            final_text = clean_answer(raw)
            trace["answer"] = final_text or None
            trace["status"] = "final_answer" if final_text else "empty_answer"
            history.append({"role": "assistant", "content": final_text})
            if verbose:
                print("\nFINAL ANSWER:", final_text or "[empty]")
            break
        if trace["tool_call_count"] + len(calls) > max_tool_calls:
            trace["status"] = "max_tool_calls_reached"
            if verbose:
                print("STOP: tool-call budget exceeded; these calls were NOT executed.")
            break
        signature = json.dumps(calls, sort_keys=True, ensure_ascii=False)
        if signature == last_signature:
            trace["status"] = "repeated_tool_request"
            if verbose:
                print("STOP: identical consecutive tool requests; avoiding a loop.")
            break
        last_signature = signature

        results = execute_calls(calls)  # READ-ONLY, allowlisted, schema-validated
        if len(results) != len(calls):
            raise RuntimeError("Tool executor returned the wrong number of results")
        turn["tool_results"] = results
        trace["tool_call_count"] += len(calls)
        if verbose:
            print("Python tool results:\n", json.dumps(results, indent=2, ensure_ascii=False))
            print("Returning these tool results to Qwen for its next turn...")
        add_exchange(history, calls, results)
    else:
        trace["status"] = "max_rounds_reached"
        if verbose:
            print("\nSTOP: maximum model turns reached without a final answer.")

    if verbose:
        print("\nRUN STATUS:", trace["status"], "| tool calls:", trace["tool_call_count"])
        if trace["status"] != "final_answer":
            print("No verified final answer was produced; inspect the trace.")
    return trace


**Questions E1 (What / How / Why)**

1. **What** does `history` contain after a tool execution? Identify where the structured tool request and JSON result are stored.
2. **How** does the next Qwen turn get access to the database result?
3. **Why** must `execute_calls` enforce an allowlist and validate schemas, even if the model was given tool descriptions?
4. **Why** do we keep `max_rounds`, `max_tool_calls`, and a repeated-call detector?
5. **What** does `max_rounds_reached` mean? Is it a successful answer? Why not?


## E2 — Check the control loop

This test deliberately substitutes **scripted fake model responses** to check Python orchestration without spending model inference time. **These assertions do NOT show that Qwen can choose the right sequence.** The real inference comes next.

Tests cover: two consecutive tool calls then a final answer, a direct response with no tool, malformed calls, repeated calls, tool-call budgets, conversation structure, and response cleanup.


In [ ]:
%%writefile e_agent_checks.py
"""Part E2 — Deterministic orchestration tests, WITHOUT A REAL LLM OR GPU.

This file uses **scripted fake model replies** ONLY to test Python's loop and
conversation message format. It does not measure Qwen tool-calling quality.
Run as `!python e_agent_checks.py`. No database mutation is performed.
"""
import json
from e_agent_loop import run_agent, add_exchange, clean_answer

SEARCH_XML = ("<tool_call><function=search_books>"
              "<parameter=genre>sci-fi</parameter>"
              "<parameter=is_bestseller>True</parameter>"
              "<parameter=max_price>16</parameter>"
              "</function></tool_call>")
RATING_XML = ("<tool_call><function=get_book_rating>"
              "<parameter=book_id>1</parameter>"
              "</function></tool_call>")


def scripted_replies(outputs):
    it = iter(outputs)
    def fake_model(history, max_new_tokens):
        return next(it)
    return fake_model


def fake_executor(calls):
    """Deterministic fixture executor, with no database connection."""
    out = []
    for call in calls:
        if call['name'] == 'search_books':
            value = [{"id": 1, "title": "Dune", "price": 14.99,
                      "genre": "sci-fi", "is_bestseller": True}]
        elif call['name'] == 'get_book_rating':
            value = {"book_id": 1, "found": True,
                     "average_rating": 4.67, "review_count": 3}
        else:
            out.append({"tool": call['name'], "status": "blocked"})
            continue
        out.append({"tool": call['name'], "status": "ok",
                    "arguments": call['arguments'], "result": value})
    return out


trace = run_agent(
    "Find a sci-fi bestseller under $16 with rating above 4.5",
    scripted_replies([SEARCH_XML, RATING_XML, "Dune: $14.99, 4.67/5 (3 reviews).<|im_end|>"]),
    fake_executor, verbose=False,
)
assert trace['status'] == 'final_answer', trace['status']
assert len(trace['turns']) == 3
assert trace['tool_call_count'] == 2
assert trace['turns'][0]['requested_tools'][0]['arguments']['is_bestseller'] is True
assert trace['turns'][1]['requested_tools'][0]['name'] == 'get_book_rating'
assert trace['answer'] == "Dune: $14.99, 4.67/5 (3 reviews)."
roles = [item['role'] for item in trace['messages']]
assert roles == ['system','user','assistant','tool','assistant','tool','assistant'], roles
assert trace['messages'][2]['tool_calls'][0]['function']['name'] == 'search_books'
assert json.loads(trace['messages'][3]['content'])['status'] == 'ok'
print("PASS: scripted two-tool chain + final answer + structured conversation history")

trace2 = run_agent("Hi!", scripted_replies(['Hello!']), fake_executor, verbose=False)
assert trace2['status'] == 'final_answer' and trace2['tool_call_count'] == 0
print("PASS: optional direct answer without any tool")

trace3 = run_agent("Find books", scripted_replies(['<tool_call>oops</tool_call>']),
                   fake_executor, verbose=False)
assert trace3['status'] == 'invalid_tool_call' and trace3['tool_call_count'] == 0
print("PASS: malformed calls are not executed")

trace4 = run_agent("Find books", scripted_replies([SEARCH_XML, SEARCH_XML]),
                   fake_executor, verbose=False)
assert trace4['status'] == 'repeated_tool_request' and trace4['tool_call_count'] == 1
print("PASS: repeated identical request is stopped")

trace5 = run_agent("Find books", scripted_replies([SEARCH_XML]),
                   fake_executor, max_rounds=1, verbose=False)
assert trace5['status'] == 'max_rounds_reached' and trace5['answer'] is None
print("PASS: max-rounds stop is not mislabeled as an answer")

trace6 = run_agent("Find books", scripted_replies([SEARCH_XML, "Done"]),
                   fake_executor, max_tool_calls=1, verbose=False)
assert trace6['tool_call_count'] == 1  # one call fits the budget
print("PASS: calls within the budget are allowed")

trace7 = run_agent("Find books", scripted_replies([SEARCH_XML + RATING_XML]),
                   fake_executor, max_tool_calls=1, verbose=False)
assert trace7['status'] == 'max_tool_calls_reached'
assert trace7['tool_call_count'] == 0
print("PASS: tool call over budget is not executed")

assert clean_answer('<think>hidden</think> Answer<|im_end|>') == 'Answer'
print("PASS: thinking / generation delimiters removed from final response")

print("\nAll deterministic Part E checks passed. Next run the REAL Qwen cell in Colab.")


In [ ]:
!python e_agent_checks.py


**Questions E2**

1. **Why** are deterministic tests useful before a GPU-backed experiment?
2. **What** is the crucial difference between the scripted tests and the live Qwen demonstration?
3. **What** properties do the scripted tests establish? Which model behaviors do they *not* establish?


## E3 — Let Qwen control the multi-step sequence (real inference)

We now run the **actual** Qwen model with the real SQLite tool executor.

The instructions make clear that book search results do not include ratings. Qwen must request a rating lookup before claiming that the result meets a rating threshold. **Do not silently insert or fabricate a missing tool call.** If Qwen skips it, this is an observation for discussion, not a success.

**Runtime:** Use `%run -i`, *not* `!python`: Part B's loaded GPU model lives in the current notebook kernel. If the runtime has restarted, rerun the relevant Part B model-loading cell first.


In [ ]:
%%writefile e_run_agent.py
"""Part E3 — Execute a REAL multi-turn Qwen3.5 + SQLite tool-calling loop.

Run `%run -i e_run_agent.py` inside the existing Colab notebook, after
`%run -i qwen_load.py`. Never run this using `!python`—that would lose the
already loaded model from Part B.

To customize: assign E_QUESTION in your notebook or in e_question.py, then
rerun `%run -i e_run_agent.py` without redownloading Qwen.
"""
from e_agent_loop import generate_qwen_reply, run_agent
from d_tool_executor import execute_requests

if "model" not in globals() or "processor" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("Run %run -i qwen_load.py first in this same notebook session")

E_QUESTION = globals().get(
    "E_QUESTION",
    "Find a sci-fi bestseller under $16 with a rating above 4.5."
)


def e_generate_reply(history, max_new_tokens):
    return generate_qwen_reply(
        history, model, processor, tokenizer, max_new_tokens=max_new_tokens
    )

E_TRACE = run_agent(
    E_QUESTION,
    generate_reply=e_generate_reply,
    execute_calls=execute_requests,
    max_rounds=5,        # up to five Qwen turns, not five Python tool calls
    max_tool_calls=8,    # allows several rating checks when multiple books match
    max_new_tokens=384,
    verbose=True,
)


In [ ]:
%run -i e_run_agent.py


**Questions E3 — inspect your actual trace**

1. **Which tool** was chosen in model turn 1? Did it match your prediction?
2. **What** exact data returned from SQLite? Where is it stored in the conversation history?
3. **Did Qwen request `get_book_rating`** before making a rating-based recommendation? If not, what failed?
4. **How many model turns** and **how many tool requests** did the full exchange require? Why are these different counts?
5. **Is** the final answer supported by the tool results? Identify supporting evidence or unsupported claims.
6. **How** could you identify a hallucinated rating using only the trace?


## E4 — Summarize the trace (no additional model generation)

We now print a compact timeline and inspect the recorded messages. This step uses the existing `E_TRACE`, so it does **not** load or call Qwen again.


In [ ]:
%%writefile e_inspect_trace.py
"""Part E4 — Summarize REAL prior model/tool turns without extra GPU inference.

Run `%run -i e_inspect_trace.py` after running e_run_agent.py.
"""
if "E_TRACE" not in globals():
    raise RuntimeError("First run %run -i e_run_agent.py")

print("USER REQUEST:", E_TRACE["question"])
print("STATUS:", E_TRACE["status"])
print("MODEL TURNS:", len(E_TRACE["turns"]))
print("TOOL REQUESTS:", E_TRACE["tool_call_count"])
print("\nTimeline:")
for turn in E_TRACE["turns"]:
    print(f"Model turn {turn['turn']}: {turn['selection_status']}")
    for call, result in zip(turn["requested_tools"], turn["tool_results"]):
        print(" ->", call["name"], "arguments=", call["arguments"])
        print(" <-", result["status"], "result=", result.get("result", result.get("error")))

looked_up_rating = any(
    item["name"] == "get_book_rating"
    for turn in E_TRACE["turns"]
    for item in turn["requested_tools"]
)
verified_rating = any(
    result.get("tool") == "get_book_rating"
    and result.get("status") == "ok"
    and isinstance(result.get("result"), dict)
    and result["result"].get("found") is True
    and result["result"].get("average_rating") is not None
    for turn in E_TRACE["turns"]
    for result in turn["tool_results"]
)
print("\nDid the agent request get_book_rating?", looked_up_rating)
print("Did the rating tool RETURN a real rating?", verified_rating)
print("Final answer:", E_TRACE["answer"] or "[not produced]")
if "rating" in E_TRACE["question"].lower() and not verified_rating:
    print("CAUTION: The rating requirement was not verified by a successful rating-tool result.")
if E_TRACE["status"] != "final_answer":
    print("CAUTION: The run ended without a final answer. See raw turns above.")


In [ ]:
%run -i e_inspect_trace.py


**Questions E4**

1. **What** evidence in the trace shows that tool outputs were supplied to the model rather than simply printed?
2. **When** should an agent stop requesting tools?
3. **How** does the code behave if a call is malformed, repeated, or exceeds its budget?
4. **What** would change if `order_book` were available? Why would a separate confirmation step be essential?


## E5 — Experiment: modify the query and compare traces

Predict the next tool sequence, then change the question by executing the file below. It only changes a Python variable. Afterwards **rerun** `%run -i e_run_agent.py` (E3) and `%run -i e_inspect_trace.py` (E4).

Additional prompts to try one at a time:

- `Find a fantasy bestseller under $15 with a rating above 4.5.`
- `What is the rating of book ID 999?`
- `Hello! What can you help me with?`
- `Find a sci-fi book under $10 with a rating above 4.9.`

**For each request, fill in a trace log:** What did you predict? Which tools were actually requested? Which results were returned? Was the final answer supported by the results? Did the loop terminate?


In [ ]:
%%writefile e_choose_question.py
"""Part E5 — Experiment: replace E_QUESTION and re-run E3."""
E_QUESTION = "Find a fantasy bestseller under $15 with a rating above 4.5."
print("New question:", E_QUESTION)
print("Now rerun: %run -i e_run_agent.py")


In [ ]:
%run -i e_choose_question.py


### Part E exit ticket

In your own words, complete the explanation:

> The model ______ a tool call, but ______ validates and executes it. The tool result is appended to the conversation as a ______ message. The next model turn may request ______ or provide a ______ answer. The loop stops when ______.

**Checkpoint for Part F:** Explain how you would introduce `get_book_reviews(book_id)` while preserving the existing tool allowlist and parameter validation. Do not implement it yet—we will do that in Part F.

**Implementation note:** The regex-based parser in Part D and this handcrafted orchestration loop are meant for instruction, not production. Production agents need stronger protocol parsing, authorization, auditing, and robustness. Also, `search_books(max_price=16)` implements **≤ $16**. Compare *under $16* with *at most $16*. Which condition does this function implement?

